# Exercise File 2 — Medium Concepts (Fresh Variation)
**Concepts:** Melt/Pivot · Custom Binning · Rolling Window · MoM % Change · 
Join/Merge · Consecutive Streak · Self-Filter · String Methods · pd.cut · Interpolation

---
## Q1 — Reshape Quarterly Headcount Report (Medium)

**Concept:** melt (wide→long) · pivot_table (long→wide)

**Problem:** HR sends a quarterly headcount table where each quarter is a column. Convert it to long format for charting, then reconstruct the wide format.

**Sample Input (wide):**

| dept    | Q1_hc | Q2_hc | Q3_hc | Q4_hc |
|---------|-------|-------|-------|-------|
| Eng     | 50    | 55    | 60    | 58    |
| Sales   | 30    | 32    | 31    | 35    |
| HR      | 10    | 10    | 12    | 11    |

**Sample Output (long):**

| dept  | quarter | headcount |
|-------|---------|-----------|
| Eng   | Q1_hc   | 50        |
| Eng   | Q2_hc   | 55        |
| ...   | ...     | ...       |

**Sample Output (reconstructed wide):** Same as original input.

In [ ]:
import pandas as pd

headcount = pd.DataFrame({
    'dept':  ['Eng','Sales','HR','Product'],
    'Q1_hc': [50, 30, 10, 15],
    'Q2_hc': [55, 32, 10, 18],
    'Q3_hc': [60, 31, 12, 20],
    'Q4_hc': [58, 35, 11, 22]
})
print('Wide:')
print(headcount)


**Concepts to use:**
1. `df.melt(id_vars=['dept'], var_name='quarter', value_name='headcount')` — wide → long.
2. `pivot_table(index='dept', columns='quarter', values='headcount', aggfunc='sum')` — long → wide.
3. `.reset_index().rename_axis(None, axis=1)` — clean column axis label.

In [ ]:
# Optimised Solution
# Dynamically infer quarter columns — any column ending with '_hc'
quarters = [c for c in headcount.columns if c.endswith('_hc')]
df_long = headcount.melt(id_vars=['dept'], value_vars=quarters, var_name='quarter', value_name='headcount')
print('Long:')
print(df_long.head(6))

df_wide = (
    df_long.pivot_table(index='dept', columns='quarter', values='headcount', aggfunc='sum')
    .reset_index().rename_axis(None, axis=1)
)
print('Reconstructed wide:')
print(df_wide)


---
## Q2 — Bin Employee Tenure into Seniority Bands (Medium)

**Concept:** pd.cut with custom bins and labels · value_counts

**Problem:** HR wants to segment employees by tenure (years at the company): <1yr → 'Newcomer', 1-3yrs → 'Junior', 3-7yrs → 'Mid', 7+ → 'Senior'. Count how many fall into each band.

**Sample Input:**

| emp_id | name    | tenure_years |
|--------|---------|--------------|
| E1     | Alice   | 0.5          |
| E2     | Bob     | 2.0          |
| E3     | Carol   | 5.5          |
| E4     | Dave    | 10.0         |
| E5     | Eve     | 1.0          |
| E6     | Frank   | 3.0          |
| E7     | Grace   | 7.0          |

**Sample Output:**

| band     | count |
|----------|-------|
| Newcomer | 1     |
| Junior   | 2     |
| Mid      | 2     |
| Senior   | 2     |

In [13]:
import pandas as pd

employees = pd.DataFrame({
    'emp_id':       ['E1','E2','E3','E4','E5','E6','E7','E8'],
    'name':         ['Alice','Bob','Carol','Dave','Eve','Frank','Grace','Hank'],
    'tenure_years': [0.5, 2.0, 5.5, 10.0, 1.0, 3.0, 7.0, 0.1]
})
print(employees)

df = employees.copy()

bins = [0,1,3,7,float('inf')]

labels = ['Newcomer','Joiner','Mid','Senior']

df_cut  = pd.cut(df['tenure_years'],bins=bins,labels=labels,include_lowest=True)

df['band'] = df_cut

# df_grouped = df.groupby('band')['emp_id'].size().reset_index().set_index('band')

df['band'].value_counts()


  emp_id   name  tenure_years
0     E1  Alice           0.5
1     E2    Bob           2.0
2     E3  Carol           5.5
3     E4   Dave          10.0
4     E5    Eve           1.0
5     E6  Frank           3.0
6     E7  Grace           7.0
7     E8   Hank           0.1


band
Newcomer    3
Joiner      2
Mid         2
Senior      1
Name: count, dtype: int64

**Concepts to use:**
1. `pd.cut(series, bins=[0,1,3,7,100], labels=['Newcomer','Junior','Mid','Senior'], right=True)`.
2. `bins` are exclusive on the left, inclusive on the right by default.
3. `.value_counts().sort_index()` — count per band, sorted by band order.

In [ ]:
# Optimised Solution
def seniority_bands(df):
    df = df.copy()
    df['band'] = pd.cut(
        df['tenure_years'],
        bins=[0, 1, 3, 7, 100],
        labels=['Newcomer','Junior','Mid','Senior'],
        right=True
    )
    print(df)
    return df['band'].value_counts().sort_index().reset_index()

print(seniority_bands(employees))


---
## Q3 — Find Users Who Ordered Every Day of a Full Week (Medium)

**Concept:** between · groupby().nunique() · boolean comparison

**Problem:** A grocery delivery app wants to reward users who placed at least one order on **every day of the week of 2024-04-01 (Mon) through 2024-04-07 (Sun)**. Return the qualifying user IDs.

**Sample Input:**

| user_id | order_date |
|---------|------------|
| U1      | 2024-04-01 |
| U1      | 2024-04-02 |
| ...     | (all 7 days) |
| U2      | 2024-04-01 |
| U2      | 2024-04-03 |
| U3      | (all 7 + duplicates) |

**Sample Output:**

| user_id |
|---------|
| U1      |
| U3      |

> U2 ❌ — missing Apr 2, 4, 5, 6, 7. U3 ✅ — 7 distinct days even with duplicates.

In [34]:
import pandas as pd

orders = pd.DataFrame({
    'user_id': (
        ['U1']*7 +
        ['U2']*3 +
        ['U3']*9   # 7 days + 2 duplicates
    ),
    'order_date': pd.to_datetime(
        ['2024-04-01','2024-04-02','2024-04-03','2024-04-04',
         '2024-04-05','2024-04-06','2024-04-07'] +   # U1: all 7 ✅
        ['2024-04-01','2024-04-03','2024-04-05'] +   # U2: only 3 ❌
        ['2024-04-01','2024-04-01','2024-04-02','2024-04-03', # U3: dups ✅
         '2024-04-04','2024-04-05','2024-04-06','2024-04-07','2024-04-07']
    )
})
print(orders.head(10))

df = orders.copy()

df =  df[df['order_date'].between('2024-04-01','2024-04-07')]

df_grouped = df.groupby('user_id')['order_date'].nunique() 

df_grouped = df_grouped[df_grouped == 7]


df_grouped.index.to_list()

# df_grouped = df.groupby('user_id')['order_date'].nunique()

# df_grouped


  user_id order_date
0      U1 2024-04-01
1      U1 2024-04-02
2      U1 2024-04-03
3      U1 2024-04-04
4      U1 2024-04-05
5      U1 2024-04-06
6      U1 2024-04-07
7      U2 2024-04-01
8      U2 2024-04-03
9      U2 2024-04-05


['U1', 'U3']

**Concepts to use:**
1. `between('2024-04-01','2024-04-07')` — filter to the target week.
2. `groupby('user_id')['order_date'].nunique()` — count distinct order days per user.
3. Filter where distinct days == 7.

In [23]:
# Optimised Solution
def daily_orderers(df):
    week = df[df['order_date'].between('2024-04-01','2024-04-07')].copy()
    week['date'] = week['order_date'].dt.date
    distinct_days = week.groupby('user_id')['date'].nunique()
    return distinct_days[distinct_days == 7].index.tolist()

print('Users who ordered every day:', daily_orderers(orders))


Users who ordered every day: ['U1', 'U3']


---
## Q4 — Monthly Active Users (MAU) % Change per App (Medium)

**Concept:** sort_values · groupby().pct_change() · first row NaN

**Problem:** A product analytics team tracks MAU per app. Compute the month-over-month % change in MAU **per app independently**.

**Sample Input:**

| app    | month   | mau  |
|--------|---------|------|
| AppA   | 2024-01 | 1000 |
| AppA   | 2024-02 | 1200 |
| AppA   | 2024-03 | 1100 |
| AppB   | 2024-01 | 500  |
| AppB   | 2024-02 | 600  |
| AppB   | 2024-03 | 550  |

**Sample Output:**

| app  | month   | mau  | mom_pct |
|------|---------|------|---------|
| AppA | 2024-01 | 1000 | NaN     |
| AppA | 2024-02 | 1200 | +20.0%  |
| AppA | 2024-03 | 1100 | -8.33%  |
| AppB | 2024-01 | 500  | NaN     |
| AppB | 2024-02 | 600  | +20.0%  |
| AppB | 2024-03 | 550  | -8.33%  |

In [56]:
import pandas as pd

mau = pd.DataFrame({
    'app':   ['AppA','AppA','AppA','AppB','AppB','AppB','AppC','AppC'],
    'month': pd.to_datetime(['2024-01','2024-02','2024-03',
                             '2024-01','2024-02','2024-03',
                             '2024-02','2024-03']),
    'mau':   [1000, 1200, 1100, 500, 600, 550, 200, 180]
})
print(mau)

df = mau.copy()
df = df.sort_values(by = ['app','month'],ascending=[True,True])

# df['pre_mau'] = df.groupby(['app'])['mau'].shift(1)

df['mom_pct'] = (df.groupby(['app'])['mau'].pct_change()*100).round(2)

df


    app      month   mau
0  AppA 2024-01-01  1000
1  AppA 2024-02-01  1200
2  AppA 2024-03-01  1100
3  AppB 2024-01-01   500
4  AppB 2024-02-01   600
5  AppB 2024-03-01   550
6  AppC 2024-02-01   200
7  AppC 2024-03-01   180


,app,month,mau,mom_pct
0,AppA,2024-01-01,1000,NaN
1,AppA,2024-02-01,1200,20.00
2,AppA,2024-03-01,1100,-8.33
3,AppB,2024-01-01,500,NaN
4,AppB,2024-02-01,600,20.00
5,AppB,2024-03-01,550,-8.33
6,AppC,2024-02-01,200,NaN
7,AppC,2024-03-01,180,-10.00


**Concepts to use:**
1. Sort by `['app','month']` first — pct_change is order-dependent.
2. `groupby('app')['mau'].pct_change() * 100` — % change within each app.
3. First row of each group is NaN — expected.

In [ ]:
# Optimised Solution
def mau_mom(df):
    df = df.sort_values(['app','month']).copy()
    df['mom_pct'] = df.groupby('app')['mau'].pct_change() * 100
    return df

print(mau_mom(mau).round(2))


---
## Q5 — Find High-Earning Drivers with 5+ Trips (Medium)

**Concept:** LEFT JOIN · groupby agg · filter

**Problem:** Join `drivers` and `trips` tables. Find drivers who have completed **5 or more trips** and show their total earnings. Include drivers with 0 trips (total_earnings = 0).

**Sample Input — drivers:**

| driver_id | name    |
|-----------|---------|
| D1        | Ali     |
| D2        | Ben     |
| D3        | Cara    |
| D4        | Dan     |

**Sample Input — trips:**

| trip_id | driver_id | fare |
|---------|-----------|------|
| T1      | D1        | 12   |
| T2      | D1        | 15   |
| ...     | (D1 has 5)|      |
| T6      | D2        | 8    |

**Sample Output:**

| driver_id | name | trip_count | total_earnings |
|-----------|------|------------|----------------|
| D1        | Ali  | 5          | 70             |

In [65]:
import pandas as pd

drivers = pd.DataFrame({
    'driver_id': ['D1','D2','D3','D4'],
    'name':      ['Ali','Ben','Cara','Dan']
})
trips = pd.DataFrame({
    'trip_id':   ['T1','T2','T3','T4','T5','T6','T7','T8'],
    'driver_id': ['D1','D1','D1','D1','D1','D2','D2','D3'],
    'fare':      [12, 15, 10, 18, 15, 8, 12, 20]
})
print(drivers)
print(trips)


merged_df = pd.merge(left=trips,right=drivers,on='driver_id')


grouped_df = merged_df.groupby('driver_id').agg(trip_count = ('trip_id','count')
                                                ,earnings = ('fare','sum')).reset_index()


grouped_df = grouped_df[grouped_df['trip_count'] > 0]

grouped_df

  driver_id  name
0        D1   Ali
1        D2   Ben
2        D3  Cara
3        D4   Dan
  trip_id driver_id  fare
0      T1        D1    12
1      T2        D1    15
2      T3        D1    10
3      T4        D1    18
4      T5        D1    15
5      T6        D2     8
6      T7        D2    12
7      T8        D3    20


,driver_id,trip_count,earnings
0,D1,5,70
1,D2,2,20
2,D3,1,20


**Concepts to use:**
1. `pd.merge(drivers, trips, on='driver_id', how='left')` — keep all drivers.
2. `groupby().agg(trip_count=('trip_id','count'), total_earnings=('fare','sum'))`.
3. Filter `trip_count >= 5`.

In [ ]:
# Optimised Solution
def high_earning_drivers(drv, trp):
    merged = drv.merge(trp, on='driver_id', how='left')
    summary = (
        merged
        .groupby(['driver_id','name'], as_index=False)
        .agg(trip_count=('trip_id','count'), total_earnings=('fare','sum'))
        .fillna(0)
    )
    return summary[summary['trip_count'] >= 5]

print(high_earning_drivers(drivers, trips))


---
## Q6 — Find Users with a 5-Consecutive-Day Workout Streak (Medium)

**Concept:** drop_duplicates · shift · cumsum streak_id · groupby streak length

**Problem:** A fitness app wants to award badges to users who worked out on **5 or more consecutive days**. Duplicate entries on the same day should be ignored.

**Sample Input:**

| user_id | workout_date |
|---------|--------------|
| U1      | 2024-01-01   |
| U1      | 2024-01-02   |
| U1      | 2024-01-03   |
| U1      | 2024-01-04   |
| U1      | 2024-01-05   |
| U2      | 2024-01-01   |
| U2      | 2024-01-03   |
| U3      | (7 days + dup) |

**Sample Output:**

| user_id |
|---------|
| U1      |
| U3      |

> U2 ❌ — gap on Jan 2. U1 ✅ exactly 5. U3 ✅ 7 days.

In [96]:
import pandas as pd

workouts = pd.DataFrame({
    'user_id': ['U1']*5 + ['U2']*3 + ['U3']*8,
    'workout_date': pd.to_datetime(
        ['2024-01-01','2024-01-02','2024-01-03','2024-01-04','2024-01-05'] +
        ['2024-01-01','2024-01-03','2024-01-05'] +
        ['2024-01-01','2024-01-01','2024-01-02','2024-01-03','2024-01-04',
         '2024-01-05','2024-01-06','2024-01-07']
    )
})
print(workouts.head(10))

df = workouts.copy()

df.drop_duplicates(inplace=True)

df = df.sort_values(by=['user_id','workout_date'])

df['date_rank'] = (df.groupby('user_id')['workout_date'].rank()).astype(int)


df['streak_date'] = df['workout_date'] - pd.to_timedelta(df['date_rank'],unit='D')


df_grouped = df.groupby('user_id')['streak_date'].count()

df_grouped = df_grouped[df_grouped >= 5]

df_grouped.tolist()

# df['date_rank'] = df.groupby('user_id')

  user_id workout_date
0      U1   2024-01-01
1      U1   2024-01-02
2      U1   2024-01-03
3      U1   2024-01-04
4      U1   2024-01-05
5      U2   2024-01-01
6      U2   2024-01-03
7      U2   2024-01-05
8      U3   2024-01-01
9      U3   2024-01-01


[5, 7]

**Concepts to use:**
1. `drop_duplicates(['user_id','workout_date'])` — one entry per user per day.
2. `groupby('user_id')['workout_date'].shift(1)` → compute gap in days.
3. `gap != 1` → streak break → `cumsum` creates a streak_id.
4. `groupby(['user_id','streak_id']).size() >= 5` → qualifying streak.

In [ ]:
# Optimised Solution
def find_5_day_streak(df):
    df = df.drop_duplicates(['user_id','workout_date']).sort_values(['user_id','workout_date'])
    df = df.copy()
    df['prev'] = df.groupby('user_id')['workout_date'].shift(1)
    df['gap']  = (df['workout_date'] - df['prev']).dt.days
    df['new_streak'] = (df['gap'] != 1) | df['gap'].isna()
    df['streak_id']  = df.groupby('user_id')['new_streak'].cumsum()
    lengths = df.groupby(['user_id','streak_id']).size().reset_index(name='len')
    return lengths[lengths['len'] >= 5]['user_id'].unique().tolist()

print('Users with 5-day streak:', find_5_day_streak(workouts))


---
## Q7 — Support Agents Who Resolved Their Own Tickets (Medium)

**Concept:** filter where two columns are equal · drop_duplicates · sort

**Problem:** In the support system, tickets can be submitted and resolved by different agents. Find agents who resolved a ticket **they themselves submitted**. Return sorted agent IDs, each appearing once.

**Sample Input:**

| ticket_id | submitted_by | resolved_by | category |
|-----------|--------------|-------------|----------|
| TK1       | A1           | A2          | Billing  |
| TK2       | A3           | A3          | Tech     |
| TK3       | A2           | A2          | Tech     |
| TK4       | A1           | A1          | Billing  |
| TK5       | A3           | A2          | Billing  |
| TK6       | A3           | A3          | Shipping |

**Sample Output:**

| agent_id |
|----------|
| A1       |
| A2       |
| A3       |

> A3 appears for TK2 and TK6 but should show only once.

In [108]:
import pandas as pd

tickets = pd.DataFrame({
    'ticket_id':    ['TK1','TK2','TK3','TK4','TK5','TK6'],
    'submitted_by': ['A1','A3','A2','A1','A3','A3'],
    'resolved_by':  ['A2','A3','A2','A1','A2','A3'],
    'category':     ['Billing','Tech','Tech','Billing','Billing','Shipping']
})
print(tickets)

df = tickets.copy()

df

df_merge = pd.merge(left=tickets,right=tickets,left_on='submitted_by',right_on='resolved_by') 

df_merge[['submitted_by_x']].drop_duplicates().rename(columns= {'submitted_by_x':'agent_id'}).reset_index(drop = True)


  ticket_id submitted_by resolved_by  category
0       TK1           A1          A2   Billing
1       TK2           A3          A3      Tech
2       TK3           A2          A2      Tech
3       TK4           A1          A1   Billing
4       TK5           A3          A2   Billing
5       TK6           A3          A3  Shipping


,agent_id
0,A1
1,A3
2,A2


**Concepts to use:**
1. Filter `submitted_by == resolved_by`.
2. `[['resolved_by']].drop_duplicates()` — each agent once.
3. Sort ascending and rename column cleanly.

In [99]:
# Optimised Solution
def self_resolved_agents(df):
    return (
        df[df['submitted_by'] == df['resolved_by']]
        [['resolved_by']]
        .drop_duplicates()
        .sort_values('resolved_by')
        .rename(columns={'resolved_by': 'agent_id'})
        .reset_index(drop=True)
    )

print(self_resolved_agents(tickets))


  agent_id
0       A1
1       A2
2       A3


---
## Q8 — Flag Suspicious Product Reviews (String Methods) (Medium)

**Concept:** str.count · str methods · boolean filters

**Problem:** Content moderation needs to flag reviews that are likely fake or spam. Flag a review if it has **3 or more exclamation marks** OR **2 or more ALL-CAPS words** (words where every character is uppercase).

**Sample Input:**

| review_id | text                               |
|-----------|------------------------------------|
| R1        | Great product! Really happy!       |
| R2        | BUY NOW! BEST DEAL EVER!!!!!       |
| R3        | it is okay                         |
| R4        | AMAZING quality, love it!!!        |
| R5        | Fast delivery! Recommended.        |

**Sample Output:**

| review_id | text                         | excl_count | caps_words | flagged |
|-----------|------------------------------|------------|------------|---------|
| R2        | BUY NOW! BEST DEAL EVER!!!!! | 5          | 4          | True    |
| R4        | AMAZING quality, love it!!!  | 3          | 1          | True    |

> R1: only 2 `!` and 0 caps words — not flagged.

In [ ]:
import pandas as pd

rev = pd.DataFrame({
    'review_id': ['R1','R2','R3','R4','R5','R6'],
    'text': [
        'Great product! Really happy!',
        'BUY NOW! BEST DEAL EVER!!!!!',
        'it is okay',
        'AMAZING quality, love it!!!',
        'Fast delivery! Recommended.',
        'WOW!! INCREDIBLE VALUE!!'
    ]
})
print(rev)


**Concepts to use:**
1. `str.count('!')` — count exclamation marks per review.
2. `str.split().apply(lambda words: sum(1 for w in words if w.isupper()))` — count ALL-CAPS words.
3. Combine conditions with `|` to flag either violation.

In [ ]:
# Optimised Solution
def flag_reviews(df):
    df = df.copy()
    df['excl_count'] = df['text'].str.count(r'!')
    df['caps_words'] = df['text'].str.split().apply(
        lambda words: sum(1 for w in words if w.strip('!.,').isupper() and len(w.strip('!.,')) > 1)
    )
    df['flagged'] = (df['excl_count'] >= 3) | (df['caps_words'] >= 2)
    return df[df['flagged']]

print(flag_reviews(rev))


---
## Q9 — Bin Restaurant Ratings into Tiers (Medium)

**Concept:** pd.cut with custom labels · groupby agg after binning

**Problem:** A food critic platform wants to categorise restaurants: ≤2.0 → 'Poor', 2.1-3.5 → 'Average', 3.6-4.5 → 'Good', 4.6-5.0 → 'Excellent'. Count and find average rating per tier.

**Sample Input:**

| restaurant_id | name        | rating |
|---------------|-------------|--------|
| R1            | Curry House | 4.8    |
| R2            | Burger Barn | 3.2    |
| R3            | Sushi Stop  | 4.4    |
| R4            | Pizza Pit   | 1.9    |
| R5            | Taco Town   | 4.7    |
| R6            | Noodle Nook | 2.8    |

**Sample Output:**

| tier      | count | avg_rating |
|-----------|-------|------------|
| Poor      | 1     | 1.9        |
| Average   | 2     | 3.0        |
| Good      | 1     | 4.4        |
| Excellent | 2     | 4.75       |

In [ ]:
import pandas as pd

restaurants = pd.DataFrame({
    'restaurant_id': ['R1','R2','R3','R4','R5','R6','R7'],
    'name':   ['Curry House','Burger Barn','Sushi Stop','Pizza Pit','Taco Town','Noodle Nook','Wrap & Roll'],
    'rating': [4.8, 3.2, 4.4, 1.9, 4.7, 2.8, 2.0]
})
print(restaurants)


**Concepts to use:**
1. `pd.cut(series, bins=[0, 2, 3.5, 4.5, 5], labels=['Poor','Average','Good','Excellent'])`.
2. `right=True` (default) — bins are (left, right], so 2.0 → 'Poor'.
3. `groupby('tier', observed=True).agg(count, avg_rating)`.

In [ ]:
# Optimised Solution
def rate_restaurants(df):
    df = df.copy()
    df['tier'] = pd.cut(
        df['rating'],
        bins=[0, 2.0, 3.5, 4.5, 5.0],
        labels=['Poor','Average','Good','Excellent'],
        right=True
    )
    return (
        df.groupby('tier', observed=True)
        .agg(count=('rating','count'), avg_rating=('rating','mean'))
        .reset_index()
    )

print(rate_restaurants(restaurants))


---
## Q10 — Interpolate Missing Hourly Weather Readings (Medium)

**Concept:** Series.interpolate(method='linear') · leading/trailing NaN behaviour

**Problem:** A weather station logs temperature every hour but some readings are missing. Fill gaps using **linear interpolation**. Leading/trailing NaNs (no neighbours on one side) should remain NaN.

**Sample Input:**

| hour | temperature |
|------|-------------|
| 0    | 15.0        |
| 1    | NaN         |
| 2    | NaN         |
| 3    | 18.0        |
| 4    | NaN         |
| 5    | 20.0        |
| 6    | NaN         |

**Sample Output:**

| hour | temperature |
|------|-------------|
| 0    | 15.0        |
| 1    | 16.0        |
| 2    | 17.0        |
| 3    | 18.0        |
| 4    | 19.0        |
| 5    | 20.0        |
| 6    | NaN         |

> Hour 6 stays NaN — it's a trailing NaN with no future known point to interpolate toward.

In [ ]:
import pandas as pd
import numpy as np

weather = pd.DataFrame({
    'hour':        [0, 1, 2, 3, 4, 5, 6],
    'temperature': [15.0, np.nan, np.nan, 18.0, np.nan, 20.0, np.nan]
})
# Edge case: leading NaN
weather_lead = pd.DataFrame({
    'hour':        [0, 1, 2, 3],
    'temperature': [np.nan, 10.0, np.nan, 14.0]
})
print(weather)


**Concepts to use:**
1. `Series.interpolate(method='linear')` — fills interior NaNs linearly.
2. Trailing NaNs (no future known value) remain NaN by default.
3. `limit_direction='both'` can extend to leading NaNs if required.

In [ ]:
# Optimised Solution
def interpolate_weather(df):
    df = df.copy()
    df['temperature'] = df['temperature'].interpolate(method='linear')
    return df

print('Standard:')
print(interpolate_weather(weather))
print('Leading NaN edge case:')
print(interpolate_weather(weather_lead))
